# 04 — Uplift Models: S-Learner, T-Learner, and X-Learner Logic

This notebook trains simple uplift models.

The main quantity is:

```text
P(conversion | treated, X) - P(conversion | control, X)
```


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import numpy as np

from uplift_marketing.data import generate_synthetic_uplift_data

from sklearn.model_selection import train_test_split

from uplift_marketing.features import infer_feature_spec, split_features_treatment_outcome
from uplift_marketing.models import SLearner, TLearner
from uplift_marketing.uplift import attach_uplift_scores


data = generate_synthetic_uplift_data(n_rows=5_000, random_state=42)
spec = infer_feature_spec(data, extra_excluded=["customer_id", "visit"])
x, treatment, y = split_features_treatment_outcome(data, spec)

x_train, x_test, t_train, t_test, y_train, y_test, raw_train, raw_test = train_test_split(
    x, treatment, y, data, test_size=0.3, random_state=42, stratify=treatment.astype(str) + "_" + y.astype(str)
)

t_learner = TLearner(spec=spec, random_state=42).fit(x_train, t_train, y_train)
p_control, p_treated = t_learner.predict_potential_outcomes(x_test)
scored = attach_uplift_scores(raw_test.reset_index(drop=True), p_control=p_control, p_treated=p_treated)
scored[["p_control", "p_treated", "uplift_score"]].describe()


## Interpretation

The T-learner estimates two potential outcome models: one for treated customers and one for control customers.

The difference between them is the estimated uplift.
